# Phase4 - Estadisticas de Comparacion

Comparacion de embeddings entre:
- `phase3 notebook 3` (baseline atencion USDJPY)
- `phase4 v2` (encoder mejorado, evaluado sobre los mismos segmentos USDJPY)

Metricas: separacion de clusters, estabilidad entre semillas y separabilidad economica en test temporal.

In [1]:
import math
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations

import torch
import torch.nn as nn

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
)
from scipy.stats import kruskal, f_oneway

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 260)


In [2]:
ROOT = Path('.')
PRICE_PATH = Path('./data/Dataset_H1_v1/parquet_h1/USDJPY.parquet')
PH3_EMB = ROOT / 'phases/phase03_segment_analysis/output_attention/USDJPY_segments_attention_embeddings.parquet'
REF_CAT = ROOT / 'phases/phase03_segment_analysis/output_extraccion/USDJPY_segments_extracted_catalog.parquet'
REF_LONG = ROOT / 'phases/phase03_segment_analysis/output_extraccion/USDJPY_segments_extracted_long.parquet'
REF_SPLIT = ROOT / 'phases/phase03_segment_analysis/output_extraccion/USDJPY_segments_split_temporal.parquet'
CKPT = ROOT / 'phases/phase04_transfer_clustering/output_transfer_EURUSD_full_v2/transfer_attention_encoder.pt'

OUT_DIR = ROOT / 'phases/phase04_transfer_clustering/output_compare_phase3_nb3_vs_phase4_v2'
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [11, 42, 77]
K = 10
H = 24
MIN_GROUP = 8


In [3]:
def preprocess_sequence_v2(close_arr: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    x = np.asarray(close_arr, dtype=float)
    if len(x) == 0:
        return np.zeros(0, dtype=np.float32)
    x = np.clip(x, eps, None)
    logp = np.log(x)
    y = logp - logp[0]
    dy = np.diff(y, prepend=y[0])
    med = np.nanmedian(dy)
    mad = np.nanmedian(np.abs(dy - med))
    scale = 1.4826 * mad if np.isfinite(mad) and mad > eps else np.nan
    if not np.isfinite(scale) or scale < eps:
        scale = np.nanstd(dy)
    if not np.isfinite(scale) or scale < eps:
        scale = np.nanstd(y)
    if not np.isfinite(scale) or scale < eps:
        scale = 1.0
    z = y / float(scale)
    z = np.clip(z, -8.0, 8.0)
    return z.astype(np.float32)


def build_padded(seqs, max_len):
    n = len(seqs)
    X = np.zeros((n, max_len), dtype=np.float32)
    M = np.zeros((n, max_len), dtype=bool)
    for i, s in enumerate(seqs):
        s = np.asarray(s, dtype=np.float32)
        if len(s) > max_len:
            s = s[-max_len:]
        ln = len(s)
        X[i, :ln] = s
        M[i, :ln] = True
    return X, M


def forward_pips(price_df, ts_vec, horizon=24, pip=0.01):
    idx = price_df['timestamp'].to_numpy(dtype='datetime64[ns]')
    close = price_df['close'].to_numpy(dtype=float)
    ts = pd.to_datetime(ts_vec, errors='coerce').to_numpy(dtype='datetime64[ns]')
    pos = np.searchsorted(idx, ts, side='left')

    out = np.full(len(ts), np.nan, dtype=float)
    for i, p0 in enumerate(pos):
        p1 = p0 + horizon
        if p0 < 0 or p1 >= len(idx):
            continue
        if idx[p0] != ts[i]:
            continue
        out[i] = (close[p1] - close[p0]) / pip
    return out


class AttentionPool(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.query = nn.Parameter(torch.randn(d_model) * 0.02)

    def forward(self, h, mask):
        scale = math.sqrt(h.shape[-1])
        logits = (h * self.query.view(1, 1, -1)).sum(dim=-1) / scale
        logits = logits.masked_fill(~mask, -1e4)
        attn = torch.softmax(logits, dim=1)
        pooled = torch.einsum('bl,bld->bd', attn, h)
        return pooled


class SegmentAttentionAutoencoder(nn.Module):
    def __init__(self, max_len, d_model, emb_dim, n_head, n_layers, ff_dim, dropout):
        super().__init__()
        self.max_len = max_len
        self.in_proj = nn.Linear(1, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_head,
            dim_feedforward=ff_dim,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.pool = AttentionPool(d_model)
        self.to_emb = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, emb_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(emb_dim, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, max_len),
        )

    def forward(self, x, mask, noise_std=0.0, mask_drop=0.0):
        _, L = x.shape
        h = self.in_proj(x.unsqueeze(-1))
        h = h + self.pos_emb[:, :L, :]
        h = self.encoder(h, src_key_padding_mask=~mask)
        pooled = self.pool(h, mask)
        z = self.to_emb(pooled)
        recon = self.decoder(z)[:, :L]
        return z, recon


def evaluate_embeddings(name, Z, meta, price_df):
    splits = meta['split'].to_numpy()
    idx_train = np.where(splits == 'train')[0]
    idx_test = np.where(splits == 'test')[0]

    scaler = StandardScaler()
    Ztr = scaler.fit_transform(Z[idx_train])
    Zall = scaler.transform(Z)
    Zte = Zall[idx_test]
    ts_test = meta.iloc[idx_test]['start_ts']

    row_metrics = []
    labels_by_seed = {}
    for s in SEEDS:
        km = KMeans(n_clusters=K, n_init=30, random_state=s)
        km.fit(Ztr)
        lb = km.predict(Zte)
        labels_by_seed[s] = lb

        if len(np.unique(lb)) >= 2 and len(lb) >= 30:
            sil = silhouette_score(Zte, lb)
            ch = calinski_harabasz_score(Zte, lb)
            db = davies_bouldin_score(Zte, lb)
        else:
            sil = np.nan
            ch = np.nan
            db = np.nan
        row_metrics.append((sil, ch, db))

    ari = []
    nmi = []
    for a, b in combinations(SEEDS, 2):
        la = labels_by_seed[a]
        lb = labels_by_seed[b]
        ari.append(adjusted_rand_score(la, lb))
        nmi.append(normalized_mutual_info_score(la, lb))

    lb = labels_by_seed[42]
    fwd = forward_pips(price_df, ts_test, horizon=H, pip=0.01)
    dft = pd.DataFrame({'cluster': lb, 'ret_fwd': fwd}).dropna()

    groups = []
    for _, g in dft.groupby('cluster'):
        if len(g) >= MIN_GROUP:
            groups.append(g['ret_fwd'].to_numpy())

    p_kw = np.nan
    p_anova = np.nan
    eta2 = np.nan
    if len(groups) >= 2:
        p_kw = float(kruskal(*groups).pvalue)
        p_anova = float(f_oneway(*groups).pvalue)
        all_vals = np.concatenate(groups)
        grand = np.mean(all_vals)
        ssb = sum(len(g) * (np.mean(g) - grand) ** 2 for g in groups)
        sst = np.sum((all_vals - grand) ** 2)
        eta2 = float(ssb / sst) if sst > 0 else np.nan

    return {
        'model': name,
        'n_segments': int(len(Z)),
        'emb_dim': int(Z.shape[1]),
        'test_silhouette_mean': float(np.nanmean([x[0] for x in row_metrics])),
        'test_ch_mean': float(np.nanmean([x[1] for x in row_metrics])),
        'test_db_mean': float(np.nanmean([x[2] for x in row_metrics])),
        'stability_ari_mean': float(np.mean(ari)),
        'stability_nmi_mean': float(np.mean(nmi)),
        'econ_kw_pvalue': p_kw,
        'econ_anova_pvalue': p_anova,
        'econ_eta2': eta2,
    }


In [4]:
# 1) Cargar metadata USDJPY (mismo universo para comparar)
cat = pd.read_parquet(REF_CAT)
lng = pd.read_parquet(REF_LONG).sort_values(['global_segment_id', 'step'])
spl = pd.read_parquet(REF_SPLIT)

cat = cat.merge(spl[['global_segment_id', 'split']], on='global_segment_id', how='left')
cat['split'] = cat['split'].fillna('train')

seq_map = {sid: g['close'].to_numpy(dtype=float) for sid, g in lng.groupby('global_segment_id', sort=False)}
rows = []
for _, r in cat.iterrows():
    sid = str(r['global_segment_id'])
    seq = seq_map.get(sid)
    if seq is None or len(seq) < 6:
        continue
    rows.append({
        'global_segment_id': sid,
        'split': str(r['split']),
        'start_ts': pd.Timestamp(r['start_ts']),
        'seq': seq,
    })

META = pd.DataFrame(rows).sort_values('start_ts').reset_index(drop=True)
print('META rows =', len(META))
print(META['split'].value_counts().to_string())


META rows = 991
split
train    695
test     149
val      147


In [5]:
# 2) Embeddings de phase3 notebook 3 (baseline)
ph3 = pd.read_parquet(PH3_EMB)
emb_cols = [c for c in ph3.columns if c.startswith('emb_')]

ph3_use = META[['global_segment_id']].merge(ph3[['global_segment_id'] + emb_cols], on='global_segment_id', how='left')
Z_PH3 = ph3_use[emb_cols].to_numpy(dtype=np.float32)

print('Z_PH3 =', Z_PH3.shape)


Z_PH3 = (991, 32)


In [6]:
# 3) Embeddings de phase4 v2 sobre los mismos segmentos USDJPY
ckpt = torch.load(CKPT, map_location='cpu')
cfg = ckpt['config']

model = SegmentAttentionAutoencoder(
    max_len=int(cfg['max_len']),
    d_model=int(cfg['d_model']),
    emb_dim=int(cfg['emb_dim']),
    n_head=int(cfg['n_head']),
    n_layers=int(cfg['n_layers']),
    ff_dim=int(cfg['ff_dim']),
    dropout=float(cfg['dropout']),
)
model.load_state_dict(ckpt['state_dict'])
model.eval()

seqs_v2 = [preprocess_sequence_v2(s) for s in META['seq']]
X, M = build_padded(seqs_v2, int(cfg['max_len']))

Z_v2 = []
with torch.no_grad():
    bs = 256
    xt = torch.from_numpy(X).float()
    mt = torch.from_numpy(M).bool()
    for i in range(0, len(xt), bs):
        z, _ = model(xt[i:i+bs], mt[i:i+bs], noise_std=0.0, mask_drop=0.0)
        Z_v2.append(z.numpy())
Z_v2 = np.vstack(Z_v2).astype(np.float32)

print('Z_v2 =', Z_v2.shape)


<python-env>/lib/python3.11/site-packages/torch/nn/modules/transformer.py:508: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Z_v2 = (991, 64)


In [7]:
# 4) Evaluacion comparativa
price = pd.read_parquet(PRICE_PATH, columns=['timestamp', 'close']).copy()
price['timestamp'] = pd.to_datetime(price['timestamp'], errors='coerce')
price['close'] = pd.to_numeric(price['close'], errors='coerce')
price = price.dropna().sort_values('timestamp').drop_duplicates('timestamp', keep='last').reset_index(drop=True)

rows = []
rows.append(evaluate_embeddings('phase3_nb3_attention_baseline', Z_PH3, META, price))
rows.append(evaluate_embeddings('phase4_v2_transfer_encoder_on_usdjpy', Z_v2, META, price))

CMP = pd.DataFrame(rows)
CMP


,model,n_segments,emb_dim,test_silhouette_mean,test_ch_mean,test_db_mean,stability_ari_mean,stability_nmi_mean,econ_kw_pvalue,econ_anova_pvalue,econ_eta2
0,phase3_nb3_attention_baseline,991,32,0.336434,142.873939,0.936069,0.772358,0.835146,1.472436e-12,2.136764e-13,0.357087
1,phase4_v2_transfer_encoder_on_usdjpy,991,64,0.484477,312.570657,0.537972,0.911404,0.895298,1.100033e-13,7.521844e-17,0.442302


In [8]:
# 5) Guardar resultados
cmp_csv = OUT_DIR / 'stats_compare_phase3_nb3_vs_phase4_v2.csv'
CMP.to_csv(cmp_csv, index=False)
print('saved:', cmp_csv)


saved: ./phases/phase04_transfer_clustering/output_compare_phase3_nb3_vs_phase4_v2/stats_compare_phase3_nb3_vs_phase4_v2.csv
